In [2]:
import requests
import pandas as pd
from datetime import datetime, timezone
import time
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.colors import BoundaryNorm
from matplotlib.cm import get_cmap
from matplotlib.patches import Patch, Rectangle
import ast
import folium
from folium.plugins import MarkerCluster
import reverse_geocoder as rg
import re
import pycountry
import os
import numpy as np
import geopandas as gpd
import fiona
import sys
from shapely.geometry import Point
from sklearn.cluster import DBSCAN
import ruptures as rpt
from haversine import haversine
import functions as own
from timezonefinder import TimezoneFinder
import zoneinfo
from scipy.stats import gaussian_kde
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from libpysal.weights import lat2W
from libpysal.weights import Queen
from libpysal.weights import DistanceBand
from esda.moran import Moran_Local, Moran
from shapely.geometry import box

Create a dataframe with all spots that are active for at least six months with at at most 14 days between any two observations

In [44]:
def persistent_dataframe_creator(df_path, max_days_between_obs, min_time_series_length):
    df = pd.read_csv(df_path)
    df["spotted_at"] = pd.to_datetime(df["spotted_at"]) # spotted_at is used because of recording and only later entering data into CrowdWater
    df["date"] = pd.to_datetime(df["spotted_at"].dt.date)

    # ignore standing water type and stream type categories because they do not require time series
    df = df[~df["Category"].isin(["standing water type", "stream type"])]

    # dates per spot and category
    spot_dates = (
        df
        .groupby(["longitude", "latitude", "Category", "date"])
        .size()
        .reset_index(name="n_obs")
    )

    def get_all_streaks(dates):
        dates = sorted(set(dates))
        streaks = []
        start = dates[0]
        prev = dates[0]

        for d in dates[1:]:
            gap = (pd.Timestamp(d) - pd.Timestamp(prev)).days
            if gap <= max_days_between_obs:
                prev = d
            else:
                duration = (pd.Timestamp(prev) - pd.Timestamp(start)).days
                if duration >= min_time_series_length:
                    streaks.append({
                        "streak_start": start,
                        "streak_end": prev,
                        "streak_days": duration
                    })
                start = d
                prev = d

        # last streak
        duration = (pd.Timestamp(prev) - pd.Timestamp(start)).days
        if duration >= min_time_series_length:
            streaks.append({
                "streak_start": start,
                "streak_end": prev,
                "streak_days": duration
            })

        return pd.DataFrame(streaks)

    streak_periods = (
        spot_dates
        .groupby(["longitude", "latitude", "Category"])["date"]
        .apply(get_all_streaks)
        .reset_index(level=3, drop=True)
        .reset_index()
    )

    persistent = streak_periods.copy().reset_index(drop=True)
    persistent["streak_start"] = pd.to_datetime(persistent["streak_start"])
    persistent["streak_end"] = pd.to_datetime(persistent["streak_end"])

    # country
    spot_country = (
        df
        .groupby(["longitude", "latitude"])["Country"]
        .first()
        .reset_index()
    )
    persistent = persistent.merge(spot_country, on=["longitude", "latitude"], how="left")

    # region
    spot_region = (
        df
        .groupby(["longitude", "latitude"]) ["Region"]
        .first()
        .reset_index()
    )
    persistent = persistent.merge(spot_region, on=["longitude", "latitude"], how="left")

    # users active during the streak at this spot
    df_merged = df.merge(
        persistent[["longitude", "latitude", "Category", "streak_start", "streak_end"]],
        on=["longitude", "latitude", "Category"],
        how="inner"
    )

    df_merged = df_merged[
        (df_merged["date"] >= df_merged["streak_start"]) &
        (df_merged["date"] <= df_merged["streak_end"])
    ]

    spot_users = (
        df_merged
        .groupby(["longitude", "latitude", "Category", "streak_start", "streak_end"])["created_by"]
        .nunique()
        .reset_index(name="n_users")
    )
    persistent = persistent.merge(spot_users, on=["longitude", "latitude", "Category", "streak_start", "streak_end"], how="left")

    spot_user_ids = (
        df_merged
        .groupby(["longitude", "latitude", "Category", "streak_start", "streak_end"])["created_by"]
        .apply(lambda x: [int(i) for i in x.unique()])
        .reset_index(name="user_ids")
    )
    persistent = persistent.merge(spot_user_ids, on=["longitude", "latitude", "Category", "streak_start", "streak_end"], how="left")

    spot_obs = (
        df_merged
        .groupby(["longitude", "latitude", "Category", "streak_start", "streak_end"])
        .size()
        .reset_index(name="n_obs")
    )
    persistent = persistent.merge(spot_obs, on=["longitude", "latitude", "Category", "streak_start", "streak_end"], how="left")

    # final table
    persistent = persistent[["latitude", "longitude", "Country", "Region", "streak_days", "streak_start", "streak_end", "n_users", "user_ids", "n_obs", "Category"]]
    persistent = persistent.sort_values("streak_days", ascending=False).reset_index(drop=True)

    print(persistent.head(20))
    print(f"number of spots with >={min_time_series_length} days consecutive activity: {len(persistent)}")

    persistent.to_csv(f"../Products/CSVs/persistent_spots_{max_days_between_obs}d.csv", index=False)

In [45]:
persistent_dataframe_creator("../CWData_clean7.csv", 14, 180) # at least every 14 days for at least half a year
persistent_dataframe_creator("../CWData_clean7.csv", 30, 365) # at least every 30 days for at least a year

C:\Users\yanni\AppData\Local\Temp\ipykernel_2652\3744915861.py:2: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 4

     latitude   longitude      Country             Region  streak_days  \
0   49.375012    8.888853      Germany  Baden-Württemberg       1854.0   
1   49.375062    8.889207      Germany  Baden-Württemberg       1822.0   
2   51.060439 -115.328485       Canada            Alberta       1763.0   
3   47.726072   13.065014      Austria           Salzburg       1410.0   
4   48.328912   16.214674      Austria   Niederösterreich       1095.0   
5   47.395523    8.730670  Switzerland             Zürich        791.0   
6   47.394939    8.733537  Switzerland             Zürich        791.0   
7   47.394939    8.733537  Switzerland             Zürich        761.0   
8   47.789612   13.068625      Austria           Salzburg        755.0   
9   48.345570   16.239102      Austria   Niederösterreich        655.0   
10  47.389403    8.561560  Switzerland             Zürich        642.0   
11  47.396384    8.567287  Switzerland             Zürich        642.0   
12  47.395096    8.570016  Switzerland

C:\Users\yanni\AppData\Local\Temp\ipykernel_2652\3744915861.py:2: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 4

     latitude   longitude         Country             Region  streak_days  \
0   47.789612   13.068625         Austria           Salzburg       2189.0   
1   47.394939    8.733537     Switzerland             Zürich       2169.0   
2   47.395523    8.730670     Switzerland             Zürich       1995.0   
3   51.060439 -115.328485          Canada            Alberta       1893.0   
4   49.375012    8.888853         Germany  Baden-Württemberg       1854.0   
5   49.375062    8.889207         Germany  Baden-Württemberg       1822.0   
6   46.990400    7.450782     Switzerland               Bern       1579.0   
7   46.991798    7.464945     Switzerland               Bern       1442.0   
8   51.574168   -0.871882  United Kingdom    Buckinghamshire       1429.0   
9   51.602487   -0.880738  United Kingdom    Buckinghamshire       1429.0   
10  51.562064   -0.867595  United Kingdom    Buckinghamshire       1429.0   
11  51.608782   -0.881946  United Kingdom    Buckinghamshire       1429.0   

Now, for both of these, show user, the spots they managed, and the weighted number of spots

In [22]:
def persistent_user(persistent, category):
    # load user_ids as lists
    persistent["user_ids"] = persistent["user_ids"].apply(ast.literal_eval)

    # collect spots per user
    user_spots = {}

    for _, row in persistent.iterrows():
        spot = (row["latitude"], row["longitude"], row["Category"], row["streak_start"], row["streak_end"])
        n_users = row["n_users"]
        for user in row["user_ids"]:
            if user not in user_spots:
                user_spots[user] = {"spots": [], "weighted_contribution": 0}
            user_spots[user]["spots"].append(spot)
            user_spots[user]["weighted_contribution"] += 1 / n_users  # percentage

    # create dataframe
    user_table = pd.DataFrame([
        {
            "user_id": user,
            "n_spots": len(data["spots"]),
            "spots": data["spots"],
            "weighted_spots": round(data["weighted_contribution"], 3)
        }
        for user, data in user_spots.items()
    ]).sort_values("weighted_spots", ascending=False).reset_index(drop=True)

    if category == "14d":
        user_table.to_csv("../Products/CSVs/persistent_users_14d.csv", index=False)
    elif category == "30d":
        user_table.to_csv("../Products/CSVs/persistent_users_30d.csv", index=False)
    else:
        raise ValueError
    return user_table

In [23]:
persistent_14d = pd.read_csv("../Products/CSVs/persistent_spots_14d.csv")
persistent_30d = pd.read_csv("../Products/CSVs/persistent_spots_30d.csv")

persistent_user_14d = persistent_user(persistent_14d, "14d")
persistent_user_30d = persistent_user(persistent_30d, "30d")

Create a timeline of the number of spots starting, ending and currently active, per year for both location sets

In [24]:
persistent_14d = pd.read_csv("../Products/CSVs/persistent_spots_14d.csv")
persistent_monthly = pd.read_csv("../Products/CSVs/persistent_spots_30d.csv")
for df in [persistent_14d, persistent_monthly]:
    df["streak_start"] = pd.to_datetime(df["streak_start"])
    df["streak_end"] = pd.to_datetime(df["streak_end"])

def calc_annual(persistent):
    results = []
    for year in range(2017, 2027):
        year_start = pd.Timestamp(f"{year}-01-01")
        year_end = pd.Timestamp(f"{year}-12-31")
        if year != 2026:
            active = persistent[(persistent["streak_start"] <= year_end) & (persistent["streak_end"] > year_end)]
        else:
            active = persistent[(persistent["streak_start"] <= year_end) & (persistent["streak_end"] >= pd.Timestamp("2026-04-01"))]
        new = persistent[(persistent["streak_start"] >= year_start) & (persistent["streak_start"] <= year_end)]
        ending = persistent[
            (persistent["streak_end"] >= year_start) &
            (persistent["streak_end"] <= year_end) &
            (persistent["streak_end"] < pd.Timestamp("2026-04-01"))  # still active spots
        ]
        results.append({"year": year, "active": len(active), "new": len(new), "ending": len(ending)})
    return pd.DataFrame(results)

df_14d = calc_annual(persistent_14d)
df_monthly = calc_annual(persistent_monthly)


fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True)

for ax, df, title in zip(axes, [df_14d, df_monthly],
                          ["a) Bi-weekly Persistent Spots (≤14 days gap, ≥180 days)",
                           "b) Monthly Persistent Spots (≤30 days gap, ≥365 days)"]):
    ax.plot(df["year"], df["active"], color="teal", linewidth=2, marker="o", label="Active Streaks")
    ax.bar(df["year"] - 0.2, df["new"], width=0.35, color="lightblue", label="New Streaks", alpha=0.8)
    ax.bar(df["year"] + 0.2, df["ending"], width=0.35, color="salmon", label="Ending Streaks", alpha=0.8)
    ax.set_ylabel("Number of Spots", fontsize=18)
    ax.set_title(title, fontsize=21, fontweight="bold", loc="left")
    ax.legend(fontsize=14)
    ax.grid(axis="y", linestyle="--", alpha=0.5)
    ax.set_axisbelow(True)
    ax.tick_params(axis='y', labelsize=14)

axes[1].set_xlabel("Year", fontsize=18)
x_labels = ["2017 (from Feb)","2018","2019","2020","2021","2022","2023","2024","2025","2026 (to Apr)"]
axes[1].set_xticks(range(2017,2027), x_labels, fontsize=14)

plt.tight_layout()
plt.savefig("../Products/persistent_spots_annual_combined.png", dpi=300, bbox_inches="tight")
plt.close()

In [43]:
df_14d

,year,active,new,ending
0,2017,1,2,1
1,2018,7,6,0
2,2019,11,11,7
3,2020,23,21,9
4,2021,49,35,9
5,2022,9,7,47
6,2023,9,5,5
7,2024,7,4,6
8,2025,12,12,7
9,2026,12,0,0


Create a maps of these two persistent spot criteria.

In [26]:
def persistent_mapper(filename):
    persistent = pd.read_csv(f"../Products/CSVs/{filename}.csv")
    world = gpd.read_file("../Borders/ne_110m_admin_0_countries/ne_110m_admin_0_countries.shp", engine="fiona")

    category_colors = {
        "physical scale": "#1418fc",
        "plastic pollution": "#fbff2b",
        "soil moisture": "#82571b",
        "standing water type": "#dd6ef0",
        "stream type": "#fc1471",
        "temporary stream": "#8c14fc",
        "virtual scale": "#14c2fc"
    }

    gdf = gpd.GeoDataFrame(
        persistent,
        geometry=gpd.points_from_xy(persistent["longitude"], persistent["latitude"]),
        crs="EPSG:4326"
    ).to_crs("+proj=robin")
    gdf = gdf.sort_values("n_users", ascending=False)

    fig, ax = plt.subplots(1, 1, figsize=(16, 8))
    
    world.to_crs("+proj=robin").plot(
        ax=ax,
        color="grey",
        edgecolor="black",
        linewidth=0.5
    )

    for cat, color in category_colors.items():
        subset = gdf[gdf["Category"] == cat]
        if len(subset) == 0:
            continue
        ax.scatter(
            subset.geometry.x,
            subset.geometry.y,
            s=np.log1p(subset["n_users"]) * 70,
            color=color,
            edgecolor="black",
            linewidth=0.3,
            alpha=0.8,
            label=cat,
            zorder=2
        )

    # legend categories
    legend_cat_handles = [
        plt.scatter([], [], s=50, color=color, edgecolor="black", linewidth=0.3, alpha=0.8, label=cat.title())
        for cat, color in category_colors.items()
        if cat in gdf["Category"].unique()
    ]
    legend_cats = ax.legend(
        handles=legend_cat_handles,
        title="Category",
        loc="lower left",
        bbox_to_anchor=(0, 0.22),
        frameon=True,
        title_fontsize=15,
        fontsize=12
    )

    # legend user counts
    user_counts = [1, 5, 10, 20]
    legend_handles = [
        plt.scatter([], [], s=np.log1p(u) * 70, color="lightgrey",
                    edgecolor="black", linewidth=0.3, alpha=0.8, label=str(u))
        for u in user_counts
    ]
    legend_users = ax.legend(
        handles=legend_handles,
        title="Number of Unique Users",
        title_fontsize=15,
        fontsize=12,
        loc="lower left",
        frameon=True,
    )
    ax.add_artist(legend_cats)  # show both legends

    if "14d" in filename:
        ax.set_title("Persistent Spots (Every 14 Days for ≥180 Days)", fontsize=18)
        outpath = "../Products/persistent_spots_map_14d.png"
    else:
        ax.set_title("Persistent Spots (Every 30 Days for ≥365 Days)", fontsize=18)
        outpath = "../Products/persistent_spots_map_30d.png"

    ax.set_axis_off()

    # Inset for Europe
    ax_inset = fig.add_axes([0.75, 0.55, 0.2, 0.3])  # [left, bottom, width, height] in figure fraction

    ax_inset.add_patch(Rectangle((0, 0), 1, 1, transform=ax_inset.transAxes,
                              facecolor="white", edgecolor="none", zorder=0))

    world.to_crs("+proj=robin").plot(
        ax=ax_inset, color="grey", edgecolor="black", linewidth=0.5, zorder=1
    )

    for cat, color in category_colors.items():
        subset = gdf[gdf["Category"] == cat]
        if len(subset) == 0:
            continue
        ax_inset.scatter(
            subset.geometry.x,
            subset.geometry.y,
            s=np.log1p(subset["n_users"]) * 70,
            color=color,
            edgecolor="black",
            linewidth=0.3,
            alpha=0.8,
            zorder=2
        )

    # extent of Europe
    ax_inset.set_xlim(-1000000, 2500000)
    ax_inset.set_ylim(3700000, 6500000)
    ax_inset.set_axis_off()
    ax_inset.set_title("Europe", fontsize=15)

    # border of inset
    rect = Rectangle((0, 0), 1, 1, transform=ax_inset.transAxes,
                     fill=False, edgecolor="black", linewidth=2, zorder=10)
    ax_inset.add_patch(rect)

    plt.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close()

In [27]:
for file in ["persistent_spots_30d", "persistent_spots_14d"]:
    persistent_mapper(file)

A few stats on the persistent spots

In [28]:
df1 = pd.read_csv("../Products/CSVs/persistent_spots_14d.csv")
df2 = pd.read_csv("../Products/CSVs/persistent_spots_30d.csv")

df1_countries = df1.groupby("Country").size().reset_index(name="n_spots").sort_values("n_spots", ascending=False)

df2_countries = df2.groupby("Country").size().reset_index(name="n_spots").sort_values("n_spots", ascending=False)

df_combined = df1_countries.merge(df2_countries, on="Country", how="outer", suffixes=("_14d", "_30d")).fillna(0).astype({"n_spots_14d": int, "n_spots_30d": int}).sort_values("Country", ascending=True)

df_combined

,Country,n_spots_14d,n_spots_30d
0,Australia,1,0
1,Austria,19,14
2,Canada,1,1
3,Chile,3,2
4,France,0,1
5,Germany,19,8
6,Ireland,0,1
7,Kyrgyzstan,1,1
8,Spain,1,0
9,Switzerland,55,53


In [30]:
df1 = pd.read_csv("../Products/CSVs/persistent_spots_14d.csv")
df2 = pd.read_csv("../Products/CSVs/persistent_spots_30d.csv")

df1_countries = df1.groupby("Category").size().reset_index(name="n_spots").sort_values("n_spots", ascending=False)

df2_countries = df2.groupby("Category").size().reset_index(name="n_spots").sort_values("n_spots", ascending=False)

df_combined = df1_countries.merge(df2_countries, on="Category", how="outer", suffixes=("_14d", "_30d")).fillna(0).astype({"n_spots_14d": int, "n_spots_30d": int}).sort_values("n_spots_14d", ascending=False)

df_combined

,Category,n_spots_14d,n_spots_30d
2,temporary stream,47,63
3,virtual scale,42,28
1,soil moisture,7,6
0,physical scale,7,5


In [40]:
df1 = pd.read_csv("../Products/CSVs/persistent_spots_14d.csv")
df2 = pd.read_csv("../Products/CSVs/persistent_spots_30d.csv")
spots1 = df1[["latitude", "longitude"]].drop_duplicates()
spots2 = df2[["latitude", "longitude"]].drop_duplicates()
spot_counts1 = df1.groupby(["latitude", "longitude"]).size().reset_index(name="n_streaks")
spot_counts2 = df2.groupby(["latitude", "longitude"]).size().reset_index(name="n_streaks")

in_both = spots1.merge(spots2, on=["latitude", "longitude"], how="inner")

print(f"Number of identical spots: {len(in_both)}")
print(f"Set 1 number of total spots: {len(df1)}")
print(f"Set 2 number of total spots: {len(df2)}")
print(f"Set 1 total unique spots: {len(spots1)}")
print(f"Set 2 total unique spots: {len(spots2)}")
print(f"Set 1 spots that are also in set 2: {len(in_both)} ({len(in_both)/len(spots1)*100:.1f}%)")
print(f"Set 2 spots that are also in set 1: {len(in_both)} ({len(in_both)/len(spots2)*100:.1f}%)")

print(spot_counts1["n_streaks"].value_counts().sort_index())
print(spot_counts2["n_streaks"].value_counts().sort_index())

Number of identical spots: 65
Set 1 number of total spots: 103
Set 2 number of total spots: 102
Set 1 total unique spots: 78
Set 2 total unique spots: 90
Set 1 spots that are also in set 2: 65 (83.3%)
Set 2 spots that are also in set 1: 65 (72.2%)
n_streaks
1    63
2     7
3     6
4     2
Name: count, dtype: int64
n_streaks
1    78
2    12
Name: count, dtype: int64
